# «Москва без машины» — карта городской мобильности

In [ ]:
import pandas as pd
import geopandas as gpd
import requests
import osmnx as ox
from shapely.geometry import LineString, MultiLineString, Polygon
from shapely.ops import polygonize, unary_union

import folium
from folium.plugins import MarkerCluster, HeatMap, Fullscreen, MiniMap

## 1. Данные с data.mos.ru

In [ ]:
datamos_api = '97f069e6-eaed-409d-95cd-96679add3f05'

def get_datamos(data_set):
    features, skip = [], 0
    while True:
        url = f'https://apidata.mos.ru/v1/datasets/{data_set}/features?api_key={datamos_api}'
        if skip:
            url += f'&$skip={skip}'
        batch = requests.get(url).json()['features']
        features += batch
        if len(batch) < 1000:
            break
        skip += 1000
    gdf = gpd.GeoDataFrame.from_features(features, crs='EPSG:4326')
    attributes = pd.DataFrame(gdf['attributes'].values.tolist(), index=gdf.index)
    return pd.concat([gdf, attributes], axis=1).drop(columns='attributes')

In [ ]:
metro = get_datamos(624)
parkings = get_datamos(916)
lanes = get_datamos(897)

len(metro), len(parkings), len(lanes)

In [ ]:
metro[['NameOfStation', 'Line', 'District', 'geometry']].head(3)

## 2. Веломаршруты из OpenStreetMap

In [ ]:
overpass_url = 'https://maps.mail.ru/osm/tools/overpass/api/interpreter'

query = '''
[out:json][timeout:120];
area["name"="Москва"]["admin_level"="4"]->.msk;
relation["route"="bicycle"](area.msk);
out geom;
'''

elements = requests.post(overpass_url, data={'data': query}).json()['elements']

rows = []
for rel in elements:
    lines = []
    for member in rel.get('members', []):
        if member['type'] == 'way' and len(member.get('geometry', [])) > 1:
            lines.append(LineString([(p['lon'], p['lat']) for p in member['geometry']]))
    if lines:
        rows.append({'name': rel['tags'].get('name', 'без названия'),
                     'geometry': MultiLineString(lines)})

routes = gpd.GeoDataFrame(rows, crs='EPSG:4326')

routes['km'] = (routes.to_crs(32637).length / 1000).round(1)
routes = routes[routes['km'] > 1].reset_index(drop=True)
routes

## 3. Границы районов

In [ ]:
query = '''
[out:json][timeout:60];
area["name"="Москва"]["admin_level"="4"]->.msk;
relation["admin_level"="8"]["boundary"="administrative"](area.msk);
out tags;
'''

elements = requests.post(overpass_url, data={'data': query}).json()['elements']
print('Районов:', len(elements))

districts = ox.geocode_to_gdf(['R' + str(e['id']) for e in elements], by_osmid=True)
districts['name'] = [e['tags']['name'] for e in elements]
districts = districts[['name', 'geometry']]

moscow = districts.union_all()
districts.head(3)

## 4. Парки и природные территории из OSM

In [ ]:
query = '''
[out:json][timeout:180];
area["name"="Москва"]["admin_level"="4"]->.msk;
way["leisure"="park"](area.msk);
out geom;
'''
ways = requests.post(overpass_url, data={'data': query}).json()['elements']

rows = []
for w in ways:
    name = w['tags'].get('name')
    if name and w['geometry'][0] == w['geometry'][-1]:
        rows.append({'name': name,
                     'geometry': Polygon([(p['lon'], p['lat']) for p in w['geometry']])})

print('Парков-контуров:', len(rows))

In [ ]:
query = '''
[out:json][timeout:180];
area["name"="Москва"]["admin_level"="4"]->.msk;
(
  relation["leisure"="park"](area.msk);
  relation["leisure"="nature_reserve"](area.msk);
  relation["boundary"="national_park"](area.msk);
  relation["boundary"="protected_area"](area.msk);
);
out geom;
'''
rels = requests.post(overpass_url, data={'data': query}).json()['elements']

for rel in rels:
    name = rel['tags'].get('name')
    lines = [LineString([(p['lon'], p['lat']) for p in m['geometry']])
             for m in rel.get('members', [])
             if m['type'] == 'way' and m.get('role') in ('outer', '') and len(m.get('geometry', [])) > 1]
    if name and lines:
        polygons = list(polygonize(unary_union(lines)))
        if polygons:
            rows.append({'name': name, 'geometry': unary_union(polygons)})

parks = gpd.GeoDataFrame(rows, crs='EPSG:4326')
parks['area_ha'] = (parks.to_crs(32637).area / 10_000).round(1)
parks = parks[parks['area_ha'] >= 5].reset_index(drop=True)

parks = gpd.clip(parks, moscow, keep_geom_type=True)

print('Парков и природных территорий от 5 га:', len(parks))
parks.sort_values('area_ha', ascending=False).head(5)[['name', 'area_ha']]

## 5. Геообработка

In [ ]:
joined = gpd.sjoin(parkings[['geometry']], districts, predicate='within')
counts = joined.groupby('name').size().reset_index(name='parkings_count')

districts = districts.merge(counts, on='name', how='left')
districts['parkings_count'] = districts['parkings_count'].fillna(0).astype(int)

districts.sort_values('parkings_count', ascending=False).head(5)[['name', 'parkings_count']]

In [ ]:
routes = gpd.clip(routes, moscow, keep_geom_type=True)
routes['km'] = (routes.to_crs(32637).length / 1000).round(1)
routes[['name', 'km']]

## 6. Сохраняем данные

In [ ]:
metro.to_file('data/metro_entrances.geojson')
parkings.to_file('data/bike_parkings.geojson')
lanes.to_file('data/bike_lanes.geojson')
routes.to_file('data/bike_routes.geojson')
districts.to_file('data/districts.geojson')
parks.to_file('data/parks.geojson')

## 7. Карта

In [ ]:
districts['geometry'] = districts.geometry.simplify(0.0005)
parks['geometry'] = parks.geometry.simplify(0.0002)
routes['geometry'] = routes.geometry.simplify(0.0002)

m = folium.Map(location=[55.75, 37.62], zoom_start=11, tiles='cartodbpositron')

choropleth = folium.Choropleth(
    geo_data=districts,
    data=districts,
    columns=['name', 'parkings_count'],
    key_on='feature.properties.name',
    fill_color='PuBu',
    fill_opacity=0.55,
    line_opacity=0.3,
    legend_name='Число велопарковок в районе',
    name='Велопарковки по районам',
).add_to(m)
choropleth.geojson.add_child(folium.features.GeoJsonTooltip(
    fields=['name', 'parkings_count'], aliases=['Район:', 'Велопарковок:']))

folium.GeoJson(
    parks,
    name='Парки и природные территории (от 5 га)',
    style_function=lambda f: {'fillColor': '#a1d99b', 'color': '#74c476',
                              'weight': 1, 'fillOpacity': 0.4},
    tooltip=folium.features.GeoJsonTooltip(fields=['name', 'area_ha'],
                                           aliases=['Название:', 'Площадь, га:']),
).add_to(m)

folium.GeoJson(
    lanes[['Name', 'Type', 'geometry']],
    name='Велодорожки',
    style_function=lambda f: {'color': '#74a9cf', 'weight': 2, 'opacity': 0.8},
    tooltip=folium.features.GeoJsonTooltip(fields=['Name', 'Type'],
                                           aliases=['Название:', 'Тип:']),
    show=False,
).add_to(m)

colors = {'Вело1': '#e6550d', 'Яуза': '#6a51a3', 'Зелёное кольцо': '#006d2c',
          'Велотрасса Крылатское': '#08519c', 'Обзорный': '#636363',
          'Ароматы мира': '#dd1c77', 'Ракета': '#8c6d31'}
routes['color'] = routes['name'].map(colors).fillna('#555555')

routes = pd.concat([routes[routes['name'] != 'Вело1'],
                    routes[routes['name'] == 'Вело1']])

folium.GeoJson(
    routes,
    name='Веломаршруты',
    style_function=lambda f: {'color': f['properties']['color'], 'weight': 4, 'opacity': 0.9,
                              'dashArray': '10 10' if f['properties']['name'] == 'Вело1' else None},
    tooltip=folium.features.GeoJsonTooltip(fields=['name', 'km'],
                                           aliases=['Маршрут:', 'Длина, км:']),
).add_to(m)

HeatMap([[p.y, p.x] for p in parkings.geometry],
        name='Велопарковки (тепловая карта)', radius=11, blur=14, show=False).add_to(m)

cluster = MarkerCluster(name='Входы станций метро', show=False).add_to(m)
for _, row in metro.iterrows():
    folium.Marker(
        [row.geometry.y, row.geometry.x],
        popup=folium.Popup(f'<b>{row["NameOfStation"]}</b><br>{row["Line"]}', max_width=250),
        icon=folium.Icon(color='darkred', icon='subway', prefix='fa'),
    ).add_to(cluster)

folium.LayerControl(collapsed=False).add_to(m)
Fullscreen().add_to(m)
MiniMap(toggle_display=True).add_to(m)

m.save('index.html')
m